# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Ranking / scoring, built on top of an underlying classification signal.
The final output editors see is a ranked review queue, not a single yes/no label — so the top-level task is ranking. But the score itself is built from a probability of decline (a classification problem underneath), the same way the starter pipeline blends best_model_probability with the baseline rule score. I'm not treating this as pure clustering (I have a target, not just groups to describe) or pure classification (a bare yes/no per page throws away the "how much worse than the next page" info an editor needs to pick where to start).

In [6]:
import pandas as pd
url = "https://raw.githubusercontent.com/MananAslamDev/ML-Stuff/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

stale = (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)
declining = (df['trend_direction'] == 'down') & (df['impressions_90d'] >= 100)
thin = (df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_90d'] >= 250)
low_ctr = (df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)

flag_count = pd.concat([stale, declining, thin, low_ctr], axis=1).sum(axis=1)
multi_flag_share = (flag_count >= 2).sum() / (flag_count >= 1).sum()
print(f"Of rows matching at least one reason code, {multi_flag_share:.1%} match 2+ codes at once.")

Of rows matching at least one reason code, 36.6% match 2+ codes at once.


## 2. Target or proxy

Proxy for now, not a true observed outcome yet: is_declining_label = trend_direction == 'down', the same label the starter pipeline uses.
This is a defined-rule proxy, not an observed future outcome — trend_direction is itself computed from trend_pct, a bucket over the current trailing-90-day window, not something that happened after a decision point. The flyrank-data skill flags this directly as the label trap, which is why trend_direction and trend_pct will never appear as model features, only as this proxy target. I'm using it this week because the starter CSV is a single flat snapshot with no daily time series. Once I move to the warehouse's fact_content_daily_performance table (Week 3, data contract), I plan to replace this with a genuine observed target: prior 90 days of features predicting decline over the next 30 days.

In [7]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(df['is_declining_label'].value_counts())
print(f"Base rate: {df['is_declining_label'].mean():.1%}")

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
Base rate: 54.2%


## 3. Success metric

Precision@50. Of the top 50 pages the queue ranks first, how many actually match the target label? This matches the real decision: an editor has capacity for roughly a fixed number of reviews, so precision at a fixed K is more honest than a threshold-free metric like accuracy. It's also the metric the starter pipeline already reports (baseline 0.240, random forest 0.740), so I can compare directly against real numbers.

In [8]:
naive_top50 = df.sort_values('impressions_90d', ascending=False).head(50)
naive_p50 = naive_top50['is_declining_label'].mean()
print(f"Naive rule (rank by impressions_90d alone), precision@50: {naive_p50:.2f}")

Naive rule (rank by impressions_90d alone), precision@50: 0.42


## 4. The unit of analysis, as a real dataframe

One row = one content item (page), summarized over its trailing 90 days, for a single client.

In [9]:
lane2_cols = ['content_id', 'client_id', 'impressions_90d', 'sessions_90d', 'clicks_90d',
              'trend_direction', 'avg_position', 'ctr', 'word_count', 'content_age_days',
              'days_since_last_update', 'is_declining_label']
lane_slice = df[lane2_cols].copy()
print(f"{len(lane_slice):,} rows, {lane_slice['content_id'].nunique():,} unique content_ids")
lane_slice.head(5)

30,000 rows, 30,000 unique content_ids


,content_id,client_id,impressions_90d,sessions_90d,clicks_90d,trend_direction,avg_position,ctr,word_count,content_age_days,days_since_last_update,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,29,down,10.6,0.76,3221.0,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,7,down,20.3,0.05,2481.0,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,down,36.5,0.09,3515.0,141,20,1
3,content_331d6c4de07b,client_19581e27de,11751,78,58,stable,6.2,0.49,NaN,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,24,down,44.0,0.13,2803.0,263,14,1


## 5. Why ML beats a fixed rule here

Two pieces of evidence, both computed above: (1) 36.6% of flagged rows match 2+ reason codes at once — a fixed rule can flag a page but can't say whether "stale + declining" should outrank "declining + thin" for a limited review slot. (2) Ranking purely by impressions_90d gets precision@50 of 0.42 — looks better than the starter's stricter baseline (0.240) on this metric alone, but that's misleading: it's a looser cut that just hands the editor high-traffic pages regardless of actual decline. The real gap is against the random forest (0.740), which combines many weak, tangled signals the way no single threshold can.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.